# GeoNUSAF Part 2 / Stage 3.5 — publish the fake pairs as a Kaggle dataset

Pulls the three finished shards from the Stage-3 Hugging Face repo, verifies them against the
interface contract, and uploads them as a private Kaggle dataset that Stage 4 mounts instantly.

**Run this on CPU.** No GPU, no accelerator. It is network and disk only.

**Before you press run:**
1. Turn **Internet ON** in the notebook settings.
2. Add-ons -> Secrets -> add `KAGGLE_USERNAME` = `sugam24` and `KAGGLE_KEY` = the `key` field
   from your `kaggle.json` (Kaggle -> Settings -> API -> Create New Token). Attach both to this
   notebook.
3. Nothing else. The Stage-3 repo is public, so no Hugging Face token is needed.

Runtime is dominated by the upload, not the download. Expect roughly 20-40 minutes end to end,
and note that Kaggle then spends a further few minutes *processing* the dataset server-side
after the CLI says it is done.

## 1. Environment

In [ ]:
import os, sys, json, glob, shutil, tarfile, subprocess, time
from pathlib import Path

!pip install -q "huggingface_hub>=0.23.0" 2>&1 | tail -1

import numpy as np
import cv2
from huggingface_hub import list_repo_files, hf_hub_download

print("python  :", sys.version.split()[0])
print("cv2     :", cv2.__version__)

# free space on the working disk -- we need room for the tarballs, the extracted PNGs, and
# the zips the Kaggle CLI builds on the way out
st = shutil.disk_usage("/kaggle/working")
print(f"working disk: {st.free/1e9:.1f} GB free of {st.total/1e9:.1f} GB")
assert st.free > 12e9, "need ~12 GB free; clear /kaggle/working first"

## 2. Configuration

`DATASET_ID` is the slug Stage 4 will mount. Once created it cannot be renamed, so pick it now
and do not change it.

`CREATE_NEW=True` makes a brand new dataset. Set it to `False` to push a **new version** of one
that already exists — same command, different verb, and Kaggle keeps the old version around.

In [ ]:
HF_REPO      = "sugam24/geonusaf-stage3-fakepairs-block-fold1"
KAGGLE_USER  = "sugam24"
DATASET_SLUG = "geonusaf-fake-pairs-block-fold1"
DATASET_ID    = f"{KAGGLE_USER}/{DATASET_SLUG}"
DATASET_TITLE = "GeoNUSAF fake pairs block fold 1"     # 6-50 chars, Kaggle enforces this

CREATE_NEW   = True          # False -> `datasets version` instead of `datasets create`
VERSION_NOTE = "initial upload of R1/R2/R3 chained fake pairs"

# expected set sizes -- the contract, asserted rather than trusted
EXPECTED = {1: 804, 2: 1608, 3: 2412}
IMG_SIZE  = 512
VALID_VALUES = set(range(6)) | {255}       # 0..5 semantic + 255 ignore

WORK = "/kaggle/working"
TARS = f"{WORK}/hf_tars"                   # downloaded shards
DEST = f"{WORK}/fp"                        # the folder that gets uploaded
os.makedirs(TARS, exist_ok=True)
os.makedirs(DEST, exist_ok=True)

print("source :", HF_REPO)
print("target :", DATASET_ID, "(private)")

## 3. Credentials

Read from Kaggle Secrets into the environment. The `kaggle` CLI looks for `KAGGLE_USERNAME` /
`KAGGLE_KEY` there before it looks for a `kaggle.json` on disk, so this is enough — nothing is
written to the filesystem and nothing lands in the notebook output.

In [ ]:
from kaggle_secrets import UserSecretsClient

us = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = us.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = us.get_secret("KAGGLE_KEY")

# a cheap authenticated call, so a bad key fails here in two seconds rather than after a
# 3 GB upload
r = subprocess.run(["kaggle", "datasets", "list", "-m", "--page-size", "1"],
                   capture_output=True, text=True)
assert r.returncode == 0, f"kaggle auth failed:\n{r.stdout}\n{r.stderr}"
print("kaggle CLI authenticated as", os.environ["KAGGLE_USERNAME"])

## 4. Pull the shards from the Hub

The repo is listed rather than hard-coded, so the tag embedded in each filename does not have to
be retyped. `chunk_R*` files are the incremental crash-recovery checkpoints written during the
Stage-3 painting run — they hold the same pairs and are deliberately skipped here.

In [ ]:
files = list_repo_files(HF_REPO)
shards = sorted(f for f in files
                if f.startswith("fake_pairs_R") and f.endswith(".tar.gz"))
extras = [f for f in files if f in ("fake_pairs_manifest.json", "README.md",
                                    "provenance.json", "split.json")]

print(f"{len(files)} files in repo; {len(shards)} shards to pull")
for s in shards: print("  ", s)
assert len(shards) == 3, f"expected 3 shards, found {shards}"

local = []
for s in shards:
    t0 = time.time()
    p = hf_hub_download(HF_REPO, s, local_dir=TARS)
    local.append(p)
    print(f"  {s}: {os.path.getsize(p)/1e6:6.0f} MB in {time.time()-t0:.0f}s")

for e in extras:
    try:
        p = hf_hub_download(HF_REPO, e, local_dir=DEST)
        print("  provenance file:", e)
    except Exception as ex:
        print(f"  ({e} not pulled: {type(ex).__name__})")

## 5. Extract

Each tarball already carries a `fake_pairs_R{r}/` prefix inside it, so extracting all three into
one destination gives exactly the three sibling directories Stage 4 expects. Nothing is renamed.

In [ ]:
for p in local:
    t0 = time.time()
    with tarfile.open(p, "r:gz") as tf:
        tf.extractall(DEST)
    print(f"  {os.path.basename(p)} -> {DEST} in {time.time()-t0:.0f}s")

# the tarballs have done their job; free the space before the CLI needs it
shutil.rmtree(TARS, ignore_errors=True)

for d in sorted(glob.glob(f"{DEST}/fake_pairs_R*")):
    n = len(os.listdir(d))
    print(f"  {os.path.basename(d)}: {n} files")
st = shutil.disk_usage(WORK)
print(f"working disk: {st.free/1e9:.1f} GB free")

## 6. Verify the contract before uploading

This is the last cheap place to catch a corrupted download or a truncated tar. A silent problem
here would not surface until a Stage-4 model trained on garbage and scored badly for reasons
nobody could explain.

Checked per set: file count matches the contract, the index range `0..n-1` is complete with no
gaps, images are `512x512x3 uint8`, masks are `512x512 uint8`, and every mask value is in
`{0..5, 255}`. Geometry and dtype are checked on every file (fast — the PNG header carries it);
pixel values are checked on a random sample, because decoding 4,824 masks in full costs minutes
and buys very little over a sample of 60.

In [ ]:
rng = np.random.default_rng(0)
ok = True

for r, n_exp in EXPECTED.items():
    d = f"{DEST}/fake_pairs_R{r}"
    imgs = glob.glob(f"{d}/genimage*.png")
    msks = glob.glob(f"{d}/genmask*.png")
    idx_i = sorted(int(Path(p).stem[len("genimage"):]) for p in imgs)
    idx_m = sorted(int(Path(p).stem[len("genmask"):])  for p in msks)

    problems = []
    if len(imgs) != n_exp:            problems.append(f"{len(imgs)} images, expected {n_exp}")
    if idx_i != idx_m:                problems.append("image and mask index sets differ")
    if idx_i != list(range(n_exp)):   problems.append("index range is not 0..n-1")

    # geometry + dtype on a sample, values on the same sample
    sample = rng.choice(n_exp, size=min(60, n_exp), replace=False)
    bad_vals = set()
    for i in sample:
        im = cv2.imread(f"{d}/genimage{i}.png", cv2.IMREAD_COLOR)
        mk = cv2.imread(f"{d}/genmask{i}.png",  cv2.IMREAD_GRAYSCALE)
        if im is None or mk is None:
            problems.append(f"pair {i} unreadable"); break
        if im.shape != (IMG_SIZE, IMG_SIZE, 3) or im.dtype != np.uint8:
            problems.append(f"image {i} is {im.shape} {im.dtype}"); break
        if mk.shape != (IMG_SIZE, IMG_SIZE) or mk.dtype != np.uint8:
            problems.append(f"mask {i} is {mk.shape} {mk.dtype}"); break
        bad_vals |= set(np.unique(mk).tolist()) - VALID_VALUES
    if bad_vals:
        problems.append(f"mask values outside the contract: {sorted(bad_vals)}")

    mb = sum(os.path.getsize(p) for p in imgs + msks) / 1e6
    if problems:
        ok = False
        print(f"R{r}: FAIL -- " + "; ".join(problems))
    else:
        print(f"R{r}: {n_exp} pairs, indices 0..{n_exp-1} complete, "
              f"512x512 uint8, values in contract, {mb:.0f} MB")

assert ok, "contract violated -- do NOT upload; re-pull the failing shard"
print("\nall three sets pass the Stage-3 interface contract")

## 7. Dataset metadata

`--dir-mode zip` zips each **subdirectory** into one archive before upload, which is what you
want with ~9,600 small files: three uploads instead of nine thousand. Kaggle unpacks them
server-side during processing, so the mounted dataset is real directories, not zips. Loose
top-level files (the manifest, the card) are uploaded as-is and stay as-is.

The dataset defaults to **private**, matching `sugam24/satellite-datasets`. Make it public later
if the paper needs it.

In [ ]:
meta = {
    "title": DATASET_TITLE,
    "id": DATASET_ID,
    "licenses": [{"name": "CC-BY-4.0"}],
}
mp = f"{DEST}/dataset-metadata.json"
json.dump(meta, open(mp, "w"), indent=2)
print(open(mp).read())

print("\nfolder to upload:")
for p in sorted(os.listdir(DEST)):
    full = f"{DEST}/{p}"
    if os.path.isdir(full):
        print(f"  {p}/  ({len(os.listdir(full))} files)  -> will be zipped")
    else:
        print(f"  {p}  ({os.path.getsize(full)/1e3:.0f} KB)")

## 8. Upload

The zipping happens locally first and is not silent about progress; the upload that follows is
where the time goes. If it fails part way, just re-run this cell — `create` is safe to retry as
long as the dataset was not actually created, and if it *was*, flip `CREATE_NEW` to `False` in
section 2 and re-run to push a version instead.

In [ ]:
verb = ["datasets", "create"] if CREATE_NEW else ["datasets", "version", "-m", VERSION_NOTE]
cmd  = ["kaggle"] + verb + ["-p", DEST, "--dir-mode", "zip"]
print(" ".join(cmd), "\n")

t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
proc.wait()
print(f"\nexit code {proc.returncode} after {(time.time()-t0)/60:.1f} min")

if proc.returncode == 0:
    print(f"\nhttps://www.kaggle.com/datasets/{DATASET_ID}")
    print("Kaggle now processes it server-side (a few minutes). The dataset shows as "
          "'processing' until that finishes -- it is not attachable before then.")
else:
    print("\nupload failed -- read the output above. The extracted pairs are still in "
          f"{DEST}, so a retry does not re-download anything.")

## 9. Verify what landed

Run this **after** the dataset finishes processing (refresh the dataset page until the file
browser appears). It lists what Kaggle actually stored, which is the only way to confirm the
server-side unzip did what it was supposed to.

If the files come back as `fake_pairs_R1.zip` rather than a directory, the unzip did not happen —
tell me and Stage 4 will extract at session start instead. Everything downstream still works,
it just costs a couple of minutes per session.

In [ ]:
r = subprocess.run(["kaggle", "datasets", "files", DATASET_ID, "--page-size", "20"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

r = subprocess.run(["kaggle", "datasets", "status", DATASET_ID],
                   capture_output=True, text=True)
print("status:", (r.stdout or r.stderr).strip())

## 10. What Stage 4 mounts

Once this is done, every Stage-4 session adds one input and reads:

```
/kaggle/input/geonusaf-fake-pairs-block-fold1/
    fake_pairs_R1/genimage0.png  genmask0.png  ...   804 pairs
    fake_pairs_R2/...                                1608 pairs
    fake_pairs_R3/...                                2412 pairs
    fake_pairs_manifest.json
    README.md
```

Alongside `/kaggle/input/satellite-datasets/satellite_dataset/` for the real tiles. Images are
**BGR on disk** and masks are single-channel `{0..5} + 255`, exactly as the part-1 loaders read
them.

Reminder from the dataset card, and it is not negotiable: these pairs are **training side only**.
They must never enter a validation or test split in any fold. The generators saw block fold 1's
train tiles.